In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

# Imports

In [ ]:
%%RecordEvent
%load_ext cudf.pandas

In [ ]:
%%RecordEvent
import numpy as np
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import time

# Load Data

In [ ]:
%%RecordEvent
### cell 0 ###

train_df = pd.read_csv('/home/dias-benchmarks/notebooks/lextoumbourou/feedback3-eda-hf-custom-trainer-sift/input/feedback-prize-english-language-learning/train.csv')
test_df = pd.read_csv('/home/dias-benchmarks/notebooks/lextoumbourou/feedback3-eda-hf-custom-trainer-sift/input/feedback-prize-english-language-learning/test.csv')

# -- STEFANOS -- Replicate Data

In [ ]:
%%RecordEvent
### cell 1 ###

factor = 10
if "IREWR_LESS_REPLICATION" in os.environ and os.environ["IREWR_LESS_REPLICATION"] == "True":
    factor = factor//5
train_df = pd.concat([train_df]*factor)
test_df = pd.concat([test_df]*factor)
train_df.info()

Let's see a row from each dataset.

In [ ]:
%%RecordEvent
### cell 2 ###

train_df.head()

In [ ]:
%%RecordEvent
### cell 3 ###

test_df.head()

Then the size of each dataset.

In [ ]:
%%RecordEvent
### cell 4 ###

len(train_df), len(test_df)

In [ ]:
%%RecordEvent
### cell 5 ###

LABEL_COLUMNS = ['cohesion', 'syntax', 'vocabulary', 'phraseology', 'grammar', 'conventions']

# Text Examples

## Random Examples

In [ ]:
%%RecordEvent
### cell 6 ###

texts = train_df.sample(frac=1, random_state=420).head(4)

## Lowest Scoring Examples

In [ ]:
%%RecordEvent
### cell 7 ###

train_df['total_score'] = train_df[LABEL_COLUMNS].sum(axis=1)
lowest_df = train_df.sort_values('total_score').head(4)

## Highest Scoring Examples

In [ ]:
%%RecordEvent
### cell 8 ###

train_df['total_score'] = train_df[LABEL_COLUMNS].sum(axis=1)
highest_df = train_df.sort_values('total_score', ascending=False).head(4)

# Text Overview

## Word Count

In [ ]:
%%RecordEvent
### cell 9 ###

train_df['word_count'] = train_df.full_text.str.split().list.len()

Mean word count:

In [ ]:
%%RecordEvent
### cell 10 ###

train_df['word_count'].mean()

Max word count:

In [ ]:
%%RecordEvent
### cell 11 ###

train_df['word_count'].max()